Don't forget to run Jupyter lab on a session with the right MNE environment activate by running the following in a terminal:

source ~/mne-python/1.8.0_0/bin/activate

In [ ]:
import numpy as np
import mne
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import cm
import matplotlib.colorbar as mcolorbar
from pathlib import Path
from tqdm import tqdm
import os
import pandas as pd

# import pyvista as pv
# pv.set_jupyter_backend('trame')
# mne.viz.set_3d_backend('notebook')

In [ ]:
def select_parcel(parcellations, go_next):
    """for each list of parcellations, select the first parcellation that is not in go_next."""
    new_parcellations = []
    for parcels in parcellations:
        added = False
        for parcel in parcels:
            if parcel not in go_next:
                new_parcellations.append(parcel)
                added = True
                break
        if not added:
            new_parcellations.append(parcels[0])  # if all parcels were in go_next, add the first one
    assert len(parcellations) == len(new_parcellations)
    return new_parcellations

In [ ]:
# sample_path = mne.datasets.sample.data_path()
# print(sample_path)
# subjects_dir = sample_path / "subjects"
# print(subjects_dir)
mne.datasets.fetch_fsaverage(subjects_dir=None)
# subject = 'fsaverage'

# Correct - Incorrect RMS

In [ ]:
contrast_path = '/home/mathieu/These/data/meg_maestro/pac_analysis/seeg/epochs_rms_all_channels/rms_contrast_results_correct_incorrect.npy'
contrast_obj = np.load(contrast_path, allow_pickle=True).item()
global_montage = mne.channels.read_dig_fif('/home/mathieu/These/data/meg_maestro/pac_analysis/seeg/epochs_rms_all_channels/global_montage.fif')
global_montage.ch_names = [str(i) for i in range(len(global_montage.ch_names))]
info = mne.create_info(global_montage.ch_names, 400., ch_types='seeg')
info.set_montage(global_montage)

tvalues = contrast_obj['tvalues']
pvalues = contrast_obj['pvalues']
parcellations = contrast_obj['parcellations']
# parcellations = [parcel[0] for parcel in parcellations]
go_next = ['Unknown', 'ctx-rh-unknown', 'ctx-lh-unknown']
parcellations = select_parcel(parcellations, go_next)

# remove channels in unknown locations
regions_to_remove = ['Right-Cerebral-White-Matter', 'Left-Cerebral-White-Matter', 'CSF', 'Unknown']
indices_to_keep = [i for i, item in enumerate(parcellations) if item not in regions_to_remove]
tvalues = [tvalues[i] for i in indices_to_keep]
pvalues = [pvalues[i] for i in indices_to_keep]

info_subset = mne.pick_info(info, sel=indices_to_keep)
print(info)
print(info_subset)

vmax = np.max(np.abs(tvalues))
norm = plt.Normalize(vmin=-vmax, vmax=vmax)
cmap = cm.seismic
# cmap = matplotlib.colors.LinearSegmentedColormap.from_list('custom_cmap', ['#ff5a5f', 'white', '#087e8b'])
sensor_colors = [cmap(norm(t)) for t in tvalues]
# for c in range(len(sensor_colors)):
#     if pvalues[c]>=0.05:
#         sensor_colors[c] = cmap(0.5)

head_fs = mne.transforms.Transform('head', 'mri',trans=None)
fig = mne.viz.plot_alignment(info_subset, trans=head_fs, subject=subject, subjects_dir=subjects_dir, surfaces='pial',
                             coord_frame='mri', show_axes=False, sensor_colors=sensor_colors)
# # Set the background color after the figure is created
fig.plotter.set_background('white')  # Use this for PyVista plots

# # Optionally render the plot if needed
# # fig.show()  # Make sure to call show() to visualize the figure

fig, ax = plt.subplots(figsize=(6, 1))
cb = mcolorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal')
cb.set_label('Contrast t-value')
# plt.show()

# Novelty detection - Retrieval RMS effects

In [ ]:
contrast_path = '/home/mathieu/These/data/meg_maestro/pac_analysis/seeg/epochs_rms_all_channels/rms_contrast_results_novelty_retrieval.npy'
contrast_obj = np.load(contrast_path, allow_pickle=True).item()
global_montage = mne.channels.read_dig_fif('/home/mathieu/These/data/meg_maestro/pac_analysis/seeg/epochs_rms_all_channels/global_montage_novelty_retrieval.fif')
global_montage.ch_names = [str(i) for i in range(len(global_montage.ch_names))]
info = mne.create_info(global_montage.ch_names, 400., ch_types='seeg')
info.set_montage(global_montage)

# tvalues = contrast_obj['tvalues']
tvalues = contrast_obj['tvalues_diff']
parcellations = contrast_obj['parcellations']
go_next = ['Unknown', 'ctx-rh-unknown', 'ctx-lh-unknown']
parcellations = select_parcel(parcellations, go_next)

# remove channels in unknown locations
regions_to_remove = ['Right-Cerebral-White-Matter', 'Left-Cerebral-White-Matter', 'CSF', 'Unknown']
indices_to_keep = [i for i, item in enumerate(parcellations) if item not in regions_to_remove]
tvalues = [tvalues[i] for i in indices_to_keep]

info_subset = mne.pick_info(info, sel=indices_to_keep)
print(info)
print(info_subset)

vmax = np.max(np.abs(tvalues))
norm = plt.Normalize(vmin=-vmax, vmax=vmax)
cmap = cm.seismic
# cmap = matplotlib.colors.LinearSegmentedColormap.from_list('custom_cmap', ['#ff5a5f', 'white', '#087e8b'])
sensor_colors = [cmap(norm(t)) for t in tvalues]
# for c in range(len(sensor_colors)):
#     if pvalues[c]>=0.05:
#         sensor_colors[c] = cmap(0.5)

head_fs = mne.transforms.Transform('head', 'mri',trans=None)
fig = mne.viz.plot_alignment(info_subset, trans=head_fs, subject=subject, subjects_dir=subjects_dir, surfaces='pial',
                             coord_frame='mri', show_axes=False, sensor_colors=sensor_colors)
# Set the background color after the figure is created
fig.plotter.set_background('white')  # Use this for PyVista plots

# Optionally render the plot if needed
# fig.show()  # Make sure to call show() to visualize the figure

fig, ax = plt.subplots(figsize=(6, 1))
cb = mcolorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal')
cb.set_label('Contrast t-value')
# plt.show()

# 2d plots

## Load data

In [ ]:
path = Path("/home/mathieu/These/data/meg_maestro/seeg_trials")
subject_list = sorted(list(path.glob('*.fif')))
del subject_list[1]  # remove Bou_Ni

subjects_channels = []
subjects_parcellations = []
subjects_montages = []
subjects_electrode_positions = []
for i, epochs_path in tqdm(enumerate(subject_list)):
    subject_name = Path(epochs_path).name[:6]
    print(subject_name)

    channels_list = []
    parcellations = []
    electrode_positions = []

    epochs = mne.read_epochs(epochs_path)
    info = epochs.info
    montage = epochs.get_montage()
    subjects_montages.append(epochs.get_montage())
    head_fs = mne.transforms.Transform('head', 'mri', trans=None)
    montage.apply_trans(head_fs)
    labels, _ = mne.get_montage_volume_labels(montage, 'fsaverage',
                                              subjects_dir=None,
                                              aseg='aparc+aseg', dist=2)

    for key, value in labels.items():
        channels_list.append(key)
        parcellations.append(value)
    go_next = ['Unknown', 'ctx-rh-unknown', 'ctx-lh-unknown']
    parcellations = select_parcel(parcellations, go_next)
    subjects_channels.append(channels_list)
    subjects_parcellations.append(parcellations)

    for ch in montage.ch_names:
            pos = montage.get_positions()['ch_pos'][ch][:3]
            electrode_positions.append(pos)
    subjects_electrode_positions.append(electrode_positions)

print('Done')

### Load montage (not needed)

In [ ]:
# global_montage = mne.channels.read_dig_fif('/home/mathieu/These/data/meg_maestro/pac_analysis/seeg/epochs_rms_all_channels/global_montage.fif')
# global_montage.ch_names = [str(i) for i in range(len(global_montage.ch_names))]
# info = mne.create_info(global_montage.ch_names, 400., ch_types='seeg')
# info.set_montage(global_montage)
# # print([info['chs'][c]['loc'][:3] for c in range(len(info.ch_names))])

## Plots

In [ ]:
# orientations = {'bottom': {'azimuth': 0, 'elevation': 180, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)},
#                 'front': {'azimuth': 90, 'elevation': 90, 'distance': 0.25, 'focalpoint': (0.0, 0.0, 0.015)},
#                 'left': {'azimuth': 180, 'elevation': 90, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)},
#                 'right': {'azimuth': 0, 'elevation': 90, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)}
#                }

orientations = {'bottom': {'azimuth': 0, 'elevation': 180, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)},
                'front': {'azimuth': 90, 'elevation': 90, 'distance': 0.25, 'focalpoint': (0.0, 0.0, 0.015)},
                'right': {'azimuth': 0, 'elevation': 90, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)},
                'left_medial': {'azimuth': 0, 'elevation': 90, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)}
               }

fig = mne.viz.create_3d_figure((1400, 1400), bgcolor=(255, 255, 255))  # 800 impacts the resolution
trans = mne.transforms.Transform('head', 'mri',trans=None)
# this is the annoying mayavi scene
fig = mne.viz.plot_alignment(
    info=info,
    subject='fsaverage',
    # surfaces=dict(brain=0.95),
    surfaces=dict(pial=0.40),
    seeg=True,
    interaction='terrain',
    trans=trans,
    fig=fig)

images = {}
electrodes = {}
for direct in orientations:
    mne.viz.set_3d_view(figure=fig, **orientations[direct])
    # here is the 3D to 2D conversion
    electrodes[direct] = []
    for montage in subjects_montages:
        xy, im = mne.viz.snapshot_brain_montage(fig, montage)
        electrodes[direct].append(xy)
    images[direct] = im  # keep last as image

### Plot all electrodes

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(30,30))

regions = ['TT', 'STG', 'MTG', 'ITG', 'TP', 'EC', 'PHC', 'HPC', 'AMY']
cmap = list(plt.get_cmap('Set3').colors)
del cmap[8]
cmap = [cmap[s] for s in range(len(regions))]
colors = cmap

for direct, ax in tqdm(zip(images, axs.flat)):
    ax.imshow(images[direct])
    # ax.set_prop_cycle(color=cmap)
    for s, electrode in enumerate(electrodes[direct]):
        if direct in ('left_medial', 'right'):
            select = 'l' in direct
            index = [ch_name for ch_name in subjects_channels[s] if (("'" in ch_name) == select)]
            electrode = {k: electrode[k] for k in index}
        # electrode = electrode[~electrode.dont_plot]
        for e in electrode:
            ax.scatter(*electrode[e], color='k', s=10)
    # for electrode in electrodes[direct]:
    #     ax.scatter(*electrodes[direct][electrode], color='k', edgecolors='k', linewidth=0.5, s=2)
    ax.set_title(direct)
    ax.axis('off')
    legend_handles = [Line2D([0], [0], marker='o', color='w', label=l,
                         markerfacecolor=c, markersize=15, markeredgecolor='k', markeredgewidth=1,)
                  for c, l in zip(cmap, regions)]
    ax.legend(handles=legend_handles, loc='best')
plt.tight_layout()
    
fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', 'all_electrodes.svg'))

### Select regions

In [ ]:
# regions = ['Left-Hippocampus', 'Right-Hippocampus']
# regions = ['Left-Hippocampus', 'Left-Amygdala', 'ctx-lh-parahippocampal', 'ctx-lh-temporalpole',
#                'ctx-lh-inferiortemporal', 'ctx-lh-entorhinal', 'Right-Hippocampus', 'Right-Amygdala',
#                'ctx-rh-parahippocampal', 'ctx-rh-temporalpole', 'ctx-rh-inferiortemporal',
#                'ctx-rh-entorhinal']  # medial temporal
regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal', 'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal', 'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala', 
           'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal', 'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal', 'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']  # ROIs

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(30, 30))

cmap = plt.get_cmap('Set3').colors
cmap = [cmap[s] for s in range(int(len(regions)/2))]
colors = cmap + cmap

for direct, ax in tqdm(zip(images, axs.flat)):
    ax.imshow(images[direct])
    n_channels = 0
    for s, electrode in enumerate(electrodes[direct]):
        index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
        current_parcellations = [subjects_parcellations[s][i] for i in range(len(subjects_parcellations[s])) if subjects_parcellations[s][i] in regions]
        n_channels += len(index)
        electrode = {k: electrode[k] for k in index}
        if direct in ('left_medial', 'right'):
            select = 'l' in direct
            current_parcellations = [current_parcellations[i] for i in range(len(current_parcellations)) if (("'" in index[i]) == select)]
            index = [ch_name for ch_name in index if (("'" in ch_name) == select)]
            electrode = {k: electrode[k] for k in index}
        for i, e in enumerate(electrode):
            # ax.scatter(*electrode[e], edgecolors='k', color=colors[regions.index(current_parcellations[i])], linewidth=0.5, s=20)
            ax.scatter(*electrode[e], edgecolors='k', color=colors[regions.index(current_parcellations[i])], linewidth=2, s=120)
    ax.set_title(direct)
    ax.axis('off')
    legend_handles = [Line2D([0], [0], marker='o', color='w', label=l,
                         markerfacecolor=c, markersize=15, markeredgecolor='k', markeredgewidth=1,)
                  for c, l in zip(colors[:int(len(colors)/2)], regions[:int(len(colors)/2)])]
    ax.legend(handles=legend_handles, loc='best')
print('N channels:', n_channels)

plt.tight_layout()
    
fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', 'subset_electrodes.svg'))

## Visu of ROIs

In [ ]:
from mne.viz import Brain

regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal',
           'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal',
           'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala',
           'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal',
           'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal',
           'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']

cmap = list(plt.get_cmap('Set3').colors)
del cmap[8]  # remove gray
cmap = [cmap[s] for s in range(int(len(regions)/2))]
colors = cmap + cmap

# Views you want
orientations_main = {
    'bottom': {'azimuth': 0, 'elevation': 180},
    'front': {'azimuth': 90, 'elevation': 90},
    'right': {'azimuth': 0, 'elevation': 90},
}
special_view = {'left_medial': {'azimuth': 0, 'elevation': 90}}  # view inside right hemi

images = {}

# --- First brain with both hemispheres for 3 views ---
brain_both = Brain(
    subject='fsaverage',
    subjects_dir=None,
    hemi='both',
    surf='pial',
    alpha=0.1,
    cortex='low_contrast',
    background='white',
    size=(2000, 2000)
)
brain_both.add_volume_labels(labels=regions, alpha=1, colors=colors, legend=False)

for view_name, cam_params in orientations_main.items():
    brain_both.show_view(**cam_params)
    img = brain_both.screenshot(mode='rgb')
    images[view_name] = img

brain_both.close()  # close to free resources

# --- Second brain with only right hemisphere for medial view ---
regions_rh = [r for r in regions if r.startswith('ctx-lh') or r.startswith('Right')]
colors_rh = colors[len(regions_rh)*-1:]  # last half

brain_rh = Brain(
    subject='fsaverage',
    subjects_dir=None,
    hemi='lh',
    surf='pial',
    alpha=0.1,
    cortex='low_contrast',
    background='white',
    size=(2000, 2000)
)
brain_rh.add_volume_labels(labels=regions_rh, alpha=1, colors=colors_rh, legend=False)

for view_name, cam_params in special_view.items():
    brain_rh.show_view(**cam_params)
    img = brain_rh.screenshot(mode='rgb')
    images[view_name] = img

brain_rh.close()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(30, 30))
for ax, view_name in zip(axs.ravel(), ['bottom', 'front', 'right', 'left_medial']):
    ax.imshow(images[view_name])
    ax.set_title(view_name.replace('_', ' '))
    ax.axis('off')

plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', 'regions_of_interest.svg'))

# RMS analysis

## Save RMS contrasts t-values per electrode

In [ ]:
rms_contrast_stats = np.load('/home/mathieu/These/data/meg_maestro/pac_analysis/seeg/epochs_rms_all_channels/rms_contrast_stats.npy', allow_pickle=True).item()
tvalues = rms_contrast_stats['tvalues']
pvalues = rms_contrast_stats['pvalues']
for tval in tvalues[0]:
    if tval is not None:
        print(tval.shape)

## Reset plotting

In [ ]:
# orientations = {'bottom': {'azimuth': 0, 'elevation': 180, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)},
#                 'front': {'azimuth': 90, 'elevation': 90, 'distance': 0.25, 'focalpoint': (0.0, 0.0, 0.015)},
#                 'left': {'azimuth': 180, 'elevation': 90, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)},
#                 'right': {'azimuth': 0, 'elevation': 90, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)}
#                }

orientations = {'bottom': {'azimuth': 0, 'elevation': 180, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)},
                'front': {'azimuth': 90, 'elevation': 90, 'distance': 0.25, 'focalpoint': (0.0, 0.0, 0.015)},
                'right': {'azimuth': 0, 'elevation': 90, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)},
                'left_medial': {'azimuth': 0, 'elevation': 90, 'distance': 0.35, 'focalpoint': (0.0, -0.02, 0.0)}
               }

fig = mne.viz.create_3d_figure((1400, 1400), bgcolor=(255, 255, 255))  # 800 impacts the resolution
trans = mne.transforms.Transform('head', 'mri',trans=None)
# this is the annoying mayavi scene
fig = mne.viz.plot_alignment(
    info=info,
    subject='fsaverage',
    # surfaces=dict(brain=0.95),
    surfaces=dict(pial=0.40),
    seeg=True,
    interaction='terrain',
    trans=trans,
    fig=fig)

images = {}
electrodes = {}
for direct in orientations:
    mne.viz.set_3d_view(figure=fig, **orientations[direct])
    # here is the 3D to 2D conversion
    electrodes[direct] = []
    for montage in subjects_montages:
        xy, im = mne.viz.snapshot_brain_montage(fig, montage)
        electrodes[direct].append(xy)
    images[direct] = im  # keep last as image

## Novelty and retrieval

In [ ]:
def plot_tval(tvals, pvals, title, use_pval=True):

    regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal', 'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal', 'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala', 
               'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal', 'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal', 'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']  # ROIs
    
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(30, 30))

    dot_size = 240 if use_pval else 120
    line_size = 3 if use_pval else 2
    
    cmap = plt.get_cmap('Spectral_r')
    t_max = 0
    for direct, ax in zip(images, axs.flat):
        for s, electrode in enumerate(electrodes[direct]):
            if tvals[s] is not None:
                index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
                current_tvalues = [tvals[s][i] for i in range(len(tvals[s])) if subjects_parcellations[s][i] in regions]
                t_max = max(t_max, np.max(np.abs(current_tvalues)))
    print(t_max)
    
    for direct, ax in tqdm(zip(images, axs.flat)):
        ax.imshow(images[direct])
        n_channels = 0
        for s, electrode in enumerate(electrodes[direct]):
            if tvals[s] is not None:
                index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
                current_parcellations = [subjects_parcellations[s][i] for i in range(len(subjects_parcellations[s])) if subjects_parcellations[s][i] in regions]
                current_tvalues = [tvals[s][i] for i in range(len(tvals[s])) if subjects_parcellations[s][i] in regions]
                current_pvalues = [pvals[s][i] for i in range(len(pvals[s])) if subjects_parcellations[s][i] in regions]
                n_channels += len(index)
                electrode = {k: electrode[k] for k in index}
                if direct in ('left_medial', 'right'):
                    select = 'l' in direct
                    current_parcellations = [current_parcellations[i] for i in range(len(current_parcellations)) if (("'" in index[i]) == select)]
                    current_tvalues = [current_tvalues[i] for i in range(len(current_tvalues)) if (("'" in index[i]) == select)]
                    current_pvalues = [current_pvalues[i] for i in range(len(current_pvalues)) if (("'" in index[i]) == select)]
                    index = [ch_name for ch_name in index if (("'" in ch_name) == select)]
                    electrode = {k: electrode[k] for k in index}
                for i, e in enumerate(electrode):
                    if (use_pval == False) | (current_pvalues[i] <= 0.05):
                        ax.scatter(*electrode[e], edgecolors='k', color=cmap(current_tvalues[i] / (2*t_max) + 0.5), linewidth=line_size, s=dot_size)
                    else:
                        ax.scatter(*electrode[e], edgecolors='k', color='k', linewidth=1, s=10)
        ax.set_title(direct)
        ax.axis('off')
    print('N channels:', n_channels)

    plt.suptitle(title, fontsize=20)
    plt.tight_layout()

    cbar_ax = fig.add_axes([0.3, 0.08, 0.4, 0.02])  # [left, bottom, width, height]
    # Create colorbar with same cmap and normalization
    norm = plt.Normalize(vmin=-t_max, vmax=t_max)
    cbar = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        cax=cbar_ax, orientation='horizontal'
    )
    
    cbar.set_label('t-value', fontsize=16)
    cbar.ax.tick_params(labelsize=12)

    thresholded = '_thresholded' if use_pval else ''
    # fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', f'rms_contrast_{title}{thresholded}.svg'))


plot_tval(tvalues[0], pvalues[0], 'Novelty', use_pval=False)
plot_tval(tvalues[0], pvalues[0], 'Novelty')
plot_tval(tvalues[1], pvalues[1], 'Retrieval', use_pval=False)
plot_tval(tvalues[1], pvalues[1], 'Retrieval')

In [ ]:
def plot_tval_abs(tvals, pvals, title, use_pval=True):

    regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal', 'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal', 'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala', 
               'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal', 'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal', 'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']  # ROIs
    
    # take abs value of tval
    for s in range(len(tvals)):
        tvals[s] = np.abs(tvals[s])
    
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(30, 30))

    dot_size = 240 if use_pval else 120
    line_size = 3 if use_pval else 2
    
    cmap = plt.get_cmap('Reds')
    t_min = 1000
    t_max = 0
    for direct, ax in zip(images, axs.flat):
        for s, electrode in enumerate(electrodes[direct]):
            if tvals[s] is not None:
                index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
                current_tvalues = [tvals[s][i] for i in range(len(tvals[s])) if subjects_parcellations[s][i] in regions]
                if use_pval:
                    current_pvalues = [pvals[s][i] for i in range(len(pvals[s])) if subjects_parcellations[s][i] in regions]
                    current_tvalues = [current_tvalues[i] for i in range(len(current_tvalues)) if current_pvalues[i] < 0.05]
                if len(current_tvalues) > 0:
                    t_min = min(t_min, np.min(current_tvalues))
                    t_max = max(t_max, np.max(current_tvalues))
    
    for direct, ax in tqdm(zip(images, axs.flat)):
        ax.imshow(images[direct])
        n_channels = 0
        for s, electrode in enumerate(electrodes[direct]):
            if tvals[s] is not None:
                index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
                current_parcellations = [subjects_parcellations[s][i] for i in range(len(subjects_parcellations[s])) if subjects_parcellations[s][i] in regions]
                current_tvalues = [tvals[s][i] for i in range(len(tvals[s])) if subjects_parcellations[s][i] in regions]
                current_pvalues = [pvals[s][i] for i in range(len(pvals[s])) if subjects_parcellations[s][i] in regions]
                n_channels += len(index)
                electrode = {k: electrode[k] for k in index}
                if direct in ('left', 'right'):
                    select = 'l' in direct
                    current_parcellations = [current_parcellations[i] for i in range(len(current_parcellations)) if (("'" in index[i]) == select)]
                    current_tvalues = [current_tvalues[i] for i in range(len(current_tvalues)) if (("'" in index[i]) == select)]
                    current_pvalues = [current_pvalues[i] for i in range(len(current_pvalues)) if (("'" in index[i]) == select)]
                    index = [ch_name for ch_name in index if (("'" in ch_name) == select)]
                    electrode = {k: electrode[k] for k in index}
                for i, e in enumerate(electrode):
                    if (use_pval == False) | (current_pvalues[i] <= 0.05):
                        ax.scatter(*electrode[e], edgecolors='k', color=cmap((current_tvalues[i] - t_min) / (t_max - t_min)), linewidth=line_size, s=dot_size)
                    else:
                        ax.scatter(*electrode[e], edgecolors='k', color='k', linewidth=1, s=10)
        ax.set_title(direct)
        ax.axis('off')
    print('N channels:', n_channels)

    plt.suptitle(title, fontsize=20)
    plt.tight_layout()

    cbar_ax = fig.add_axes([0.3, 0.08, 0.4, 0.02])  # [left, bottom, width, height]
    # Create colorbar with same cmap and normalization
    norm = plt.Normalize(vmin=-t_min, vmax=t_max)
    cbar = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        cax=cbar_ax, orientation='horizontal'
    )
    
    cbar.set_label('t-value', fontsize=16)
    cbar.ax.tick_params(labelsize=12)

    thresholded = '_thresholded' if use_pval else ''
    fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', f'rms_contrast_{title}{thresholded}_rescaled.svg'))


# plot_tval_abs(tvalues[0], pvalues[0], 'Novelty', use_pval=False)
plot_tval_abs(tvalues[0], pvalues[0], 'Novelty')
# plot_tval_abs(tvalues[1], pvalues[1], 'Retrieval', use_pval=False)
plot_tval_abs(tvalues[1], pvalues[1], 'Retrieval')

In [ ]:
def plot_signif_novelty_and_retrieval(pvals):

    regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal', 'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal', 'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala', 
               'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal', 'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal', 'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']  # ROIs
    
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(30, 30))

    dot_size = 240
    dot_size = 300
    line_size = 5

    # cmap for parcellations
    cmap = list(plt.get_cmap('Set3').colors)
    del cmap[8]
    cmap = [cmap[s] for s in range(int(len(regions)/2))]
    colors = cmap + cmap
    
    for direct, ax in tqdm(zip(images, axs.flat)):
        ax.imshow(images[direct])
        n_channels = 0
        for s, electrode in enumerate(electrodes[direct]):
            assert pvals[0] is not None
            assert pvals[1] is not None
            index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
            current_parcellations = [subjects_parcellations[s][i] for i in range(len(subjects_parcellations[s])) if subjects_parcellations[s][i] in regions]
            current_pvalues_novelty = [pvals[0][s][i] for i in range(len(pvals[0][s])) if subjects_parcellations[s][i] in regions]
            current_pvalues_retrieval = [pvals[1][s][i] for i in range(len(pvals[1][s])) if subjects_parcellations[s][i] in regions]
            n_channels += len(index)
            electrode = {k: electrode[k] for k in index}
            if direct in ('left_medial', 'right'):
                select = 'l' in direct
                current_parcellations = [current_parcellations[i] for i in range(len(current_parcellations)) if (("'" in index[i]) == select)]
                current_pvalues_novelty = [current_pvalues_novelty[i] for i in range(len(current_pvalues_novelty)) if (("'" in index[i]) == select)]
                current_pvalues_retrieval = [current_pvalues_retrieval[i] for i in range(len(current_pvalues_retrieval)) if (("'" in index[i]) == select)]
                index = [ch_name for ch_name in index if (("'" in ch_name) == select)]
                electrode = {k: electrode[k] for k in index}
            for i, e in enumerate(electrode):
                if (current_pvalues_novelty[i] <= 0.05) & (current_pvalues_retrieval[i] <= 0.05):
                    ax.scatter(*electrode[e], color=colors[regions.index(current_parcellations[i])], edgecolors='#6859D9', linewidth=line_size, s=dot_size, zorder=2)
                elif (current_pvalues_novelty[i] <= 0.05):
                    ax.scatter(*electrode[e], color=colors[regions.index(current_parcellations[i])], edgecolors='#FF1100', linewidth=line_size, s=dot_size, zorder=2)
                elif (current_pvalues_retrieval[i] <= 0.05):
                    ax.scatter(*electrode[e], color=colors[regions.index(current_parcellations[i])], edgecolors='#3B2AF5', linewidth=line_size, s=dot_size, zorder=2)
                else:
                    ax.scatter(*electrode[e], edgecolors='k', color='k', linewidth=1, s=10, zorder=1)
        ax.set_title(direct)
        ax.axis('off')
    print('N channels:', n_channels)

    plt.tight_layout()

    fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', f'rms_contrast_significant_novelty_retrieval.svg'))


plot_signif_novelty_and_retrieval(pvalues)

## Novelty vs retrieval

In [ ]:
regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal', 'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal', 'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala', 
           'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal', 'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal', 'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']  # ROIs
    
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(30, 30))

cmap = plt.get_cmap('Spectral_r')
t_max = 0
for direct, ax in zip(images, axs.flat):
    for s, electrode in enumerate(electrodes[direct]):
        if (tvalues[0][s] is not None) & (tvalues[1][s] is not None):
            index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
            current_tvalues = [tvalues[1][s][i] - tvalues[0][s][i] for i in range(len(tvalues[0][s])) if subjects_parcellations[s][i] in regions]
            t_max = max(t_max, np.max(np.abs(current_tvalues)))
print(t_max)

for direct, ax in tqdm(zip(images, axs.flat)):
    ax.imshow(images[direct])
    n_channels = 0
    for s, electrode in enumerate(electrodes[direct]):
        if (tvalues[0][s] is not None) & (tvalues[1][s] is not None):
            index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
            current_parcellations = [subjects_parcellations[s][i] for i in range(len(subjects_parcellations[s])) if subjects_parcellations[s][i] in regions]
            current_tvalues = [tvalues[1][s][i] - tvalues[0][s][i] for i in range(len(tvalues[0][s])) if subjects_parcellations[s][i] in regions]
            n_channels += len(index)
            electrode = {k: electrode[k] for k in index}
            if direct in ('left', 'right'):
                select = 'l' in direct
                current_parcellations = [current_parcellations[i] for i in range(len(current_parcellations)) if (("'" in index[i]) == select)]
                current_tvalues = [current_tvalues[i] for i in range(len(current_tvalues)) if (("'" in index[i]) == select)]
                index = [ch_name for ch_name in index if (("'" in ch_name) == select)]
                electrode = {k: electrode[k] for k in index}
            for i, e in enumerate(electrode):
                ax.scatter(*electrode[e], edgecolors='k', color=cmap(current_tvalues[i] / (2*t_max) + 0.5), linewidth=2, s=120)
    ax.set_title(direct)
    ax.axis('off')
print('N channels:', n_channels)

plt.suptitle('Retrieval - Novelty', fontsize=20)
plt.tight_layout()

cbar_ax = fig.add_axes([0.3, 0.08, 0.4, 0.02])  # [left, bottom, width, height]
# Create colorbar with same cmap and normalization
norm = plt.Normalize(vmin=-t_max, vmax=t_max)
cbar = plt.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cbar_ax, orientation='horizontal'
)
cbar.set_label('Retrieval - Novelty (t-value)', fontsize=16)
cbar.ax.tick_params(labelsize=12)

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', 'rms_contrast_retrieval_vs_novelty.svg'))

# Representation analysis

In [ ]:
representation_stats = np.load('/home/mathieu/These/data/meg_maestro/representation_analysis/seeg/corr_and_distance/representation_stats.npy', allow_pickle=True).item()
representation_stats.keys()

In [ ]:
def plot_regression_params(params, pvals, title, use_pval=True):

    regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal', 'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal', 'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala', 
               'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal', 'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal', 'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']  # ROIs
    
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(30, 30))

    dot_size = 240 if use_pval else 120
    line_size = 3 if use_pval else 2
    
    cmap = plt.get_cmap('Spectral_r')
    t_max = 0
    for direct, ax in zip(images, axs.flat):
        for s, electrode in enumerate(electrodes[direct]):
            if params[s] is not None:
                index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
                current_params = [params[s][i] for i in range(len(params[s])) if subjects_parcellations[s][i] in regions]
                t_max = max(t_max, np.max(np.abs(current_params)))
    print(t_max)
    
    for direct, ax in tqdm(zip(images, axs.flat)):
        ax.imshow(images[direct])
        n_channels = 0
        for s, electrode in enumerate(electrodes[direct]):
            if params[s] is not None:
                index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
                current_parcellations = [subjects_parcellations[s][i] for i in range(len(subjects_parcellations[s])) if subjects_parcellations[s][i] in regions]
                current_params = [params[s][i] for i in range(len(params[s])) if subjects_parcellations[s][i] in regions]
                current_pvalues = [pvals[s][i] for i in range(len(pvals[s])) if subjects_parcellations[s][i] in regions]
                n_channels += len(index)
                electrode = {k: electrode[k] for k in index}
                if direct in ('left', 'right'):
                    select = 'l' in direct
                    current_parcellations = [current_parcellations[i] for i in range(len(current_parcellations)) if (("'" in index[i]) == select)]
                    current_params = [current_params[i] for i in range(len(current_params)) if (("'" in index[i]) == select)]
                    current_pvalues = [current_pvalues[i] for i in range(len(current_pvalues)) if (("'" in index[i]) == select)]
                    index = [ch_name for ch_name in index if (("'" in ch_name) == select)]
                    electrode = {k: electrode[k] for k in index}
                for i, e in enumerate(electrode):
                    if (use_pval == False) | (current_pvalues[i] <= 0.05):
                        ax.scatter(*electrode[e], edgecolors='k', color=cmap(current_params[i] / (2*t_max) + 0.5), linewidth=line_size, s=dot_size)
                    else:
                        ax.scatter(*electrode[e], edgecolors='k', color='k', linewidth=1, s=10)
        ax.set_title(direct)
        ax.axis('off')
    print('N channels:', n_channels)

    plt.suptitle(title, fontsize=20)
    plt.tight_layout()

    cbar_ax = fig.add_axes([0.3, 0.08, 0.4, 0.02])  # [left, bottom, width, height]
    # Create colorbar with same cmap and normalization
    norm = plt.Normalize(vmin=-t_max, vmax=t_max)
    cbar = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        cax=cbar_ax, orientation='horizontal'
    )
    
    cbar.set_label('Regression params', fontsize=16)
    cbar.ax.tick_params(labelsize=12)

    thresholded = '_thresholded' if use_pval else ''
    fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', f'regression_{title}{thresholded}.svg'))


plot_regression_params(representation_stats['intrastim_correct_params'], representation_stats['intrastim_correct_pvalues'], 'Correct ~ intrastim correlation', use_pval=False)
plot_regression_params(representation_stats['intrastim_surprise_params'], representation_stats['intrastim_surprise_pvalues'], 'Intrastim correlation ~ surprise', use_pval=False)
plot_regression_params(representation_stats['interstim_correct_params'], representation_stats['interstim_correct_pvalues'], 'Correct ~ interstim distance', use_pval=False)
plot_regression_params(representation_stats['interstim_surprise_params'], representation_stats['interstim_surprise_pvalues'], 'Interstim distance ~ surprise', use_pval=False)

In [ ]:
def plot_regression_params_categorical(pvals, title):

    regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal', 'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal', 'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala', 
               'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal', 'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal', 'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']  # ROIs
    
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(30, 30))

    dot_size = 300
    line_size = 5

    cmap = list(plt.get_cmap('Set3').colors)
    del cmap[8]
    cmap = [cmap[s] for s in range(int(len(regions)/2))]
    colors = cmap + cmap
    
    for direct, ax in tqdm(zip(images, axs.flat)):
        ax.imshow(images[direct])
        n_channels = 0
        for s, electrode in enumerate(electrodes[direct]):
            assert pvals[0] is not None
            assert pvals[1] is not None
            index = [subjects_channels[s][i] for i in range(len(subjects_channels[s])) if subjects_parcellations[s][i] in regions]  # select regions
            current_parcellations = [subjects_parcellations[s][i] for i in range(len(subjects_parcellations[s])) if subjects_parcellations[s][i] in regions]
            current_pvalues_novelty = [pvals[0][s][i] for i in range(len(pvals[0][s])) if subjects_parcellations[s][i] in regions]
            current_pvalues_retrieval = [pvals[1][s][i] for i in range(len(pvals[1][s])) if subjects_parcellations[s][i] in regions]
            n_channels += len(index)
            electrode = {k: electrode[k] for k in index}
            if direct in ('left_medial', 'right'):
                select = 'l' in direct
                current_parcellations = [current_parcellations[i] for i in range(len(current_parcellations)) if (("'" in index[i]) == select)]
                current_pvalues_novelty = [current_pvalues_novelty[i] for i in range(len(current_pvalues_novelty)) if (("'" in index[i]) == select)]
                current_pvalues_retrieval = [current_pvalues_retrieval[i] for i in range(len(current_pvalues_retrieval)) if (("'" in index[i]) == select)]
                index = [ch_name for ch_name in index if (("'" in ch_name) == select)]
                electrode = {k: electrode[k] for k in index}
            for i, e in enumerate(electrode):
                if (current_pvalues_novelty[i] <= 0.05) & (current_pvalues_retrieval[i] <= 0.05):
                    ax.scatter(*electrode[e], color=colors[regions.index(current_parcellations[i])], edgecolors='#02C20C', linewidth=line_size, s=dot_size, zorder=2)
                elif (current_pvalues_novelty[i] <= 0.05):
                    ax.scatter(*electrode[e], color=colors[regions.index(current_parcellations[i])], edgecolors='#FF1100', linewidth=line_size, s=dot_size, zorder=2)
                elif (current_pvalues_retrieval[i] <= 0.05):
                    ax.scatter(*electrode[e], color=colors[regions.index(current_parcellations[i])], edgecolors='#3B2AF5', linewidth=line_size, s=dot_size, zorder=2)
                else:
                    ax.scatter(*electrode[e], edgecolors='k', color='k', linewidth=1, s=10, zorder=1)
        ax.set_title(direct)
        ax.axis('off')
    print('N channels:', n_channels)

    plt.tight_layout()
    
    fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations', f'regression_{title}_intra_inter.svg'))


plot_regression_params_categorical([representation_stats['intrastim_correct_pvalues'], representation_stats['interstim_correct_pvalues']], 'Correct')
plot_regression_params_categorical([representation_stats['intrastim_surprise_pvalues'], representation_stats['interstim_surprise_pvalues']], 'Surprise')

Try combination

In [ ]:
regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal', 
           'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal', 
           'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala', 
           'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal', 
           'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal', 
           'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']

# Generate colors for regions
cmap = plt.get_cmap('Set3').colors
cmap = [cmap[i % len(cmap)] for i in range(len(regions)//2)]
colors = cmap + cmap  # Match left/right

# Create Brain instance (PyVista backend)
brain = Brain(subject='fsaverage',
              subjects_dir=None,
              hemi='both',
              surf='pial',
              alpha=0.1,
              cortex='low_contrast',
              background='white',
              size=(1400, 1400))

# Add labeled regions
brain.add_volume_labels(labels=regions,
                        alpha=0.7,
                        colors=colors,
                        legend=True)

electrode_positions = np.concatenate(subjects_electrode_positions, axis=0)  # shape (N, 3)
print(electrode_positions.shape)

# Add all electrode positions as black dots
brain.add_foci(electrode_positions,
               coords_as_verts=False,
               hemi='lh',
               color='black',
               scale_factor=1.5,  # control dot size
               resolution=20,
               name='Electrodes')

orientations = {
    'bottom': {'azimuth': 0, 'elevation': 180},
    'front': {'azimuth': 90, 'elevation': 90},
    'left': {'azimuth': 180, 'elevation': 90},
    'right': {'azimuth': 0, 'elevation': 90}
}

# Dictionary to store 2D images
images = {}

for view_name, cam_params in orientations.items():
    brain.show_view(**cam_params)
    img = brain.screenshot(mode='rgb')  # numpy array (H, W, 3)
    images[view_name] = img

    # Optional: Save image
    # plt.imsave(f'brain_{view_name}.png', img)

# Linear vs exp fits across regions

In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize

In [ ]:
from mne.viz import Brain

def setup_3d_plot(colors, regions):

    # Views you want
    orientations_main = {
        'bottom': {'azimuth': 0, 'elevation': 180},
        'front': {'azimuth': 90, 'elevation': 90},
        'right': {'azimuth': 0, 'elevation': 90},
    }
    special_view = {'left_medial': {'azimuth': 0, 'elevation': 90}}  # view inside right hemi
    
    images = {}
    
    # --- First brain with both hemispheres for 3 views ---
    brain_both = Brain(
        subject='fsaverage',
        subjects_dir=None,
        hemi='both',
        surf='pial',
        alpha=0.1,
        cortex='low_contrast',
        background='white',
        size=(2000, 2000)
    )
    brain_both.add_volume_labels(labels=regions, alpha=1, colors=colors, legend=False)
    
    for view_name, cam_params in orientations_main.items():
        brain_both.show_view(**cam_params)
        img = brain_both.screenshot(mode='rgb')
        images[view_name] = img
    
    brain_both.close()  # close to free resources
    
    # --- Second brain with only right hemisphere for medial view ---
    regions_rh = [r for r in regions if r.startswith('ctx-lh') or r.startswith('Right')]
    colors_rh = colors[len(regions_rh)*-1:]  # last half
    
    brain_rh = Brain(
        subject='fsaverage',
        subjects_dir=None,
        hemi='lh',
        surf='pial',
        alpha=0.1,
        cortex='low_contrast',
        background='white',
        size=(2000, 2000)
    )
    brain_rh.add_volume_labels(labels=regions_rh, alpha=1, colors=colors_rh, legend=False)
    
    for view_name, cam_params in special_view.items():
        brain_rh.show_view(**cam_params)
        img = brain_rh.screenshot(mode='rgb')
        images[view_name] = img
    
    brain_rh.close()

    return images

In [ ]:
# Load region proportion of best exp fit
proportion_df = pd.read_csv("/home/mathieu/These/data/meg_maestro/representation_analysis/seeg/corr_and_distance/linear_vs_exp.csv")
print(proportion_df)

In [ ]:
# --- function to normalize region names same way as before
def normalize_region(r):
    if r.startswith(("ctx-lh-", "ctx-rh-")):
        return r.split("-", 2)[-1]
    if r.startswith(("Left-", "Right-")):
        return r.split("-", 1)[-1]
    return r

regions = ['ctx-lh-transversetemporal', 'ctx-lh-superiortemporal', 'ctx-lh-middletemporal',
           'ctx-lh-inferiortemporal', 'ctx-lh-temporalpole', 'ctx-lh-entorhinal',
           'ctx-lh-parahippocampal', 'Left-Hippocampus', 'Left-Amygdala',
           'ctx-rh-transversetemporal', 'ctx-rh-superiortemporal', 'ctx-rh-middletemporal',
           'ctx-rh-inferiortemporal', 'ctx-rh-temporalpole', 'ctx-rh-entorhinal',
           'ctx-rh-parahippocampal', 'Right-Hippocampus', 'Right-Amygdala']

## AIC similarity

In [ ]:
metric = "aic_similarity"

region_value_dict = dict(
    zip(proportion_df["region_norm"], proportion_df[metric])
)

# --- get values for each plotted region
values = []
for r in regions:
    norm_name = normalize_region(r)
    values.append(region_value_dict.get(norm_name, np.nan))

values = np.array(values)

# --- normalize values for colormap
norm = Normalize(vmin=np.min(values), vmax=np.max(values))  # proportions already 0–1
cmap = cm.get_cmap("Oranges")     # choose your colormap

colors = [cmap(norm(v)) if not np.isnan(v) else (0.8,0.8,0.8,1) for v in values]

images = setup_3d_plot(colors, regions)

fig, axs = plt.subplots(2, 2, figsize=(30, 30))
for ax, view_name in zip(axs.ravel(), ['bottom', 'front', 'right', 'left_medial']):
    ax.imshow(images[view_name])
    ax.set_title(view_name.replace('_', ' '))
    ax.axis('off')

plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations/exp_vs_linear', f'{metric}.svg'))

In [ ]:
# colorbar
fig, ax = plt.subplots(figsize=(5, 1))

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])  # required

cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")

cbar.set_label("Proportion of best exponential fits", fontsize=12)
cbar.ax.tick_params(labelsize=10)
plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations/exp_vs_linear', f'{metric}_colorbar.svg'))

## AIC separability

In [ ]:
metric = "aic_separability"

region_value_dict = dict(
    zip(proportion_df["region_norm"], proportion_df[metric])
)

# --- get values for each plotted region
values = []
for r in regions:
    norm_name = normalize_region(r)
    values.append(region_value_dict.get(norm_name, np.nan))

values = np.array(values)

# --- normalize values for colormap
norm = Normalize(vmin=np.min(values), vmax=np.max(values))  # proportions already 0–1
cmap = cm.get_cmap("Oranges")     # choose your colormap

colors = [cmap(norm(v)) if not np.isnan(v) else (0.8,0.8,0.8,1) for v in values]

images = setup_3d_plot(colors, regions)

fig, axs = plt.subplots(2, 2, figsize=(30, 30))
for ax, view_name in zip(axs.ravel(), ['bottom', 'front', 'right', 'left_medial']):
    ax.imshow(images[view_name])
    ax.set_title(view_name.replace('_', ' '))
    ax.axis('off')

plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations/exp_vs_linear', f'{metric}.svg'))

In [ ]:
# colorbar
fig, ax = plt.subplots(figsize=(5, 1))

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])  # required

cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")

cbar.set_label("Proportion of best exponential fits", fontsize=12)
cbar.ax.tick_params(labelsize=10)
plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations/exp_vs_linear', f'{metric}_colorbar.svg'))

## logL similarity

In [ ]:
metric = "logL_similarity"

region_value_dict = dict(
    zip(proportion_df["region_norm"], proportion_df[metric])
)

# --- get values for each plotted region
values = []
for r in regions:
    norm_name = normalize_region(r)
    values.append(region_value_dict.get(norm_name, np.nan))

values = np.array(values)

# --- normalize values for colormap
norm = Normalize(vmin=np.min(values), vmax=np.max(values))  # proportions already 0–1
cmap = cm.get_cmap("Oranges")     # choose your colormap

colors = [cmap(norm(v)) if not np.isnan(v) else (0.8,0.8,0.8,1) for v in values]

images = setup_3d_plot(colors, regions)

fig, axs = plt.subplots(2, 2, figsize=(30, 30))
for ax, view_name in zip(axs.ravel(), ['bottom', 'front', 'right', 'left_medial']):
    ax.imshow(images[view_name])
    ax.set_title(view_name.replace('_', ' '))
    ax.axis('off')

plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations/exp_vs_linear', f'{metric}.svg'))

In [ ]:
# colorbar
fig, ax = plt.subplots(figsize=(5, 1))

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])  # required

cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")

cbar.set_label("Proportion of best exponential fits", fontsize=12)
cbar.ax.tick_params(labelsize=10)
plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations/exp_vs_linear', f'{metric}_colorbar.svg'))

## logL separability

In [ ]:
metric = "logL_separability"

region_value_dict = dict(
    zip(proportion_df["region_norm"], proportion_df[metric])
)

# --- get values for each plotted region
values = []
for r in regions:
    norm_name = normalize_region(r)
    values.append(region_value_dict.get(norm_name, np.nan))

values = np.array(values)

# --- normalize values for colormap
norm = Normalize(vmin=np.min(values), vmax=np.max(values))  # proportions already 0–1
cmap = cm.get_cmap("Oranges")     # choose your colormap

colors = [cmap(norm(v)) if not np.isnan(v) else (0.8,0.8,0.8,1) for v in values]

images = setup_3d_plot(colors, regions)

fig, axs = plt.subplots(2, 2, figsize=(30, 30))
for ax, view_name in zip(axs.ravel(), ['bottom', 'front', 'right', 'left_medial']):
    ax.imshow(images[view_name])
    ax.set_title(view_name.replace('_', ' '))
    ax.axis('off')

plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations/exp_vs_linear', f'{metric}.svg'))

In [ ]:
# colorbar
fig, ax = plt.subplots(figsize=(5, 1))

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])  # required

cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")

cbar.set_label("Proportion of best exponential fits", fontsize=12)
cbar.ax.tick_params(labelsize=10)
plt.tight_layout()
plt.show()

fig.savefig(os.path.join('/home/mathieu/These/data/meg_maestro/seeg_visualizations/exp_vs_linear', f'{metric}_colorbar.svg'))